In [1]:
# pip install xarray netcdf4

In [2]:
import xarray as xr
import numpy as np
import os
import glob
from functools import reduce

folder = "raw_data"
files = sorted(glob.glob(os.path.join(folder, "*.nc")))
print("Found", len(files), "files\n")

Found 10 files



In [3]:
# ------------------------- STEP 1: VARIABLE MAP -------------------------

print("VARIABLE MAP (file → variables):\n")

for f in files:
    ds = xr.open_dataset(f)
    vars_in_file = list(ds.data_vars.keys())
    print(f"{os.path.basename(f)}  →  {vars_in_file}")

VARIABLE MAP (file → variables):

chl.nc  →  ['chl']
currents.nc  →  ['uo', 'vo']
kd.nc  →  ['kd']
nutrients.nc  →  ['fe', 'no3', 'po4', 'si']
o2.nc  →  ['o2']
ph.nc  →  ['ph']
so.nc  →  ['so']
spco2.nc  →  ['spco2']
thetao.nc  →  ['thetao']
wo.nc  →  ['wo']


In [4]:
# ------------------------- STEP 2: PREPROCESS -------------------------

def preprocess(ds, filename):
    print(f"--- Preprocessing {filename} ---")

    # --------------------------------------------------
    # 0) ADD MISSING DEPTH FOR FILES LIKE spco2.nc
    # --------------------------------------------------
    if "depth" not in ds.coords:
        print(f"  -> depth missing. Adding depth = 0.5")
        ds = ds.expand_dims({"depth": [0.5]})

    # --------------------------------------------------
    # 1) ROUND COORDINATES
    # --------------------------------------------------
    ds = ds.assign_coords({
        "depth":     np.round(ds["depth"].values, 1),
        "latitude":  np.round(ds["latitude"].values, 2),
        "longitude": np.round(ds["longitude"].values, 2),
    })

    # TRUNCATE TIME TO DAY
    if "time" in ds.coords:
        new_time = np.array(ds["time"].values, dtype="datetime64[D]")
        ds = ds.assign_coords(time=new_time)

    # --------------------------------------------------
    # 2) CHECK FOR INVALID DEPTH VALUES (≠ 0.5)
    # --------------------------------------------------
    depth_values = ds["depth"].values
    invalid_depths = depth_values[depth_values != 0.5]

    if len(invalid_depths) > 0:
        print(f"  -> WARNING: Found {len(invalid_depths)} depth values not equal to 0.5:")
        print(f"       Invalid depths: {invalid_depths.tolist()}")
    else:
        print(f"  -> All depth values are valid (=0.5)")

    # --------------------------------------------------
    # 3) DETECT DUPLICATES
    # --------------------------------------------------
    df = ds.to_dataframe().reset_index()

    key_cols = ["time", "depth", "latitude", "longitude"]

    # Count duplicates
    dup_mask = df.duplicated(subset=key_cols, keep=False)
    dup_count = df[dup_mask].shape[0]

    print(f"  -> Duplicate coordinate entries after rounding: {dup_count}")

    # --------------------------------------------------
    # 4) AVERAGE OUT DUPLICATES
    # --------------------------------------------------
    df = df.groupby(key_cols).mean().reset_index()

    # Convert back to xarray
    ds_clean = df.set_index(key_cols).to_xarray()

    print(f"  -> After averaging, unique grid cells: {ds_clean.to_dataframe().shape[0]}")

    return ds_clean

In [5]:
# ------------------------- STEP 3: LOAD + PREPROCESS -------------------------

datasets = []
print("Preprocessing all files...")

for f in files:
    print("\n Opening:", os.path.basename(f))
    ds_clean = xr.open_dataset(f, chunks={"time": 1})

    ds_clean = preprocess(ds_clean, os.path.basename(f))

    datasets.append(ds_clean)

Preprocessing all files...

 Opening: chl.nc
--- Preprocessing chl.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 163761

 Opening: currents.nc
--- Preprocessing currents.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 1343433

 Opening: kd.nc
--- Preprocessing kd.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 163761

 Opening: nutrients.nc
--- Preprocessing nutrients.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 163761

 Opening: o2.nc
--- Preprocessing o2.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 163761

 Opening: ph.nc
--- Pre

In [6]:
# ------------------------- STEP 4: INNER JOIN COORDS -------------------------

def intersect_coord(coord):
    arrs = [ds[coord].values for ds in datasets]
    common = reduce(lambda a, b: np.intersect1d(a, b, assume_unique=True), arrs)
    base = datasets[0][coord].values
    mask = np.isin(base, common)
    return base[mask]


print("\nComputing INNER JOIN on coordinates...")

common_time  = intersect_coord("time")
common_depth = intersect_coord("depth")
common_lat   = intersect_coord("latitude")
common_lon   = intersect_coord("longitude")

print("\nAFTER INNER JOIN:")
print("  time:", len(common_time))
print("  depth:", len(common_depth))
print("  latitude:", len(common_lat))
print("  longitude:", len(common_lon))

if (
    len(common_time) == 0 or
    len(common_depth) == 0 or
    len(common_lat) == 0 or
    len(common_lon) == 0
):
    raise RuntimeError("Inner join produced empty coordinates! Aborting.")


Computing INNER JOIN on coordinates...

AFTER INNER JOIN:
  time: 741
  depth: 1
  latitude: 13
  longitude: 17


In [7]:
# ------------------------- STEP 5: SUBSET -------------------------

aligned = []
print("Subsetting every dataset to common coords...\n")

for i, ds in enumerate(datasets):
    fname = os.path.basename(files[i])
    print("Subsetting:", fname)
    ds2 = ds.sel(
        time=common_time,
        depth=common_depth,
        latitude=common_lat,
        longitude=common_lon,
        method=None
    )
    aligned.append(ds2)

Subsetting every dataset to common coords...

Subsetting: chl.nc
Subsetting: currents.nc
Subsetting: kd.nc
Subsetting: nutrients.nc
Subsetting: o2.nc
Subsetting: ph.nc
Subsetting: so.nc
Subsetting: spco2.nc
Subsetting: thetao.nc
Subsetting: wo.nc


In [8]:
# ------------------------- STEP 6: MERGE -------------------------

print("\nMerging all variables...\n")

merged = xr.merge(aligned, combine_attrs="override")

merged = merged.sortby("time")


Merging all variables...



In [9]:
# ------------------------- STEP 7: SAVE OUTPUT -------------------------

output = "data.nc"
print("Saving final merged dataset to:", output)

merged.to_netcdf(output)

print("\nDONE!")
print("Saved:", output)
print("\nFinal dataset summary:\n")
merged

Saving final merged dataset to: data.nc

DONE!
Saved: data.nc

Final dataset summary:



<xarray.Dataset> Size: 9MB
Dimensions:    (time: 741, depth: 1, latitude: 13, longitude: 17)
Coordinates:
  * time       (time) datetime64[s] 6kB 2023-11-15 2023-11-16 ... 2025-11-24
  * depth      (depth) float32 4B 0.5
  * latitude   (latitude) float32 52B 20.0 20.25 20.5 20.75 ... 22.5 22.75 23.0
  * longitude  (longitude) float32 68B 87.0 87.25 87.5 87.75 ... 90.5 90.75 91.0
Data variables: (12/14)
    chl        (time, depth, latitude, longitude) float32 655kB 0.281 ... nan
    uo         (time, depth, latitude, longitude) float32 655kB -0.006259 ......
    vo         (time, depth, latitude, longitude) float32 655kB -0.02203 ... nan
    kd         (time, depth, latitude, longitude) float32 655kB 0.05391 ... nan
    fe         (time, depth, latitude, longitude) float32 655kB 0.001012 ... nan
    no3        (time, depth, latitude, longitude) float32 655kB 1.868 ... nan
    ...         ...
    o2         (time, depth, latitude, longitude) float32 655kB 213.7 ... nan
    ph         (time, depth, latitude, longitude) float32 655kB 8.144 ... nan
    so         (time, depth, latitude, longitude) float32 655kB 25.94 ... nan
    spco2      (time, depth, latitude, longitude) float32 655kB 29.71 ... nan
    thetao     (time, depth, latitude, longitude) float32 655kB 28.08 ... nan
    wo         (time, depth, latitude, longitude) float32 655kB 9.378e-08 ......

In [11]:
import xarray as xr
import pandas as pd

# Load merged dataset
merged = xr.open_dataset("data.nc")

# Convert to DataFrame
df = merged.to_dataframe().reset_index()

df

,time,depth,latitude,longitude,chl,uo,vo,kd,fe,no3,po4,si,o2,ph,so,spco2,thetao,wo
0,2023-11-15,0.5,20.0,87.00,0.281021,-0.006259,-0.022032,0.053912,0.001012,1.868117,0.000773,1.615477,213.700699,8.144088,25.942343,29.705456,28.077726,9.377943e-08
1,2023-11-15,0.5,20.0,87.25,0.307757,-0.239613,-0.096364,0.056723,0.000788,0.651579,0.002323,1.524284,213.016098,8.151818,25.696281,29.530020,28.035089,1.243761e-06
2,2023-11-15,0.5,20.0,87.50,0.361767,-0.274904,-0.155237,0.061546,0.000665,0.060814,0.013914,1.535927,209.882004,8.116321,30.314779,32.050991,28.162968,4.969929e-06
3,2023-11-15,0.5,20.0,87.75,0.269444,-0.050030,0.156805,0.052620,0.000643,0.023415,0.045637,1.560925,207.318329,8.058519,32.787670,36.222424,28.436258,-7.740124e-07
4,2023-11-15,0.5,20.0,88.00,0.260333,-0.017039,0.178411,0.052023,0.000686,0.022156,0.056481,1.668919,206.241653,8.051982,32.575329,36.790691,28.268316,-6.153559e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163756,2025-11-24,0.5,23.0,90.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163757,2025-11-24,0.5,23.0,90.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163758,2025-11-24,0.5,23.0,90.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163759,2025-11-24,0.5,23.0,90.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df = df.dropna(how="any")

df

,time,depth,latitude,longitude,chl,uo,vo,kd,fe,no3,po4,si,o2,ph,so,spco2,thetao,wo
0,2023-11-15,0.5,20.00,87.00,0.281021,-0.006259,-0.022032,0.053912,0.001012,1.868117,0.000773,1.615477,213.700699,8.144088,25.942343,29.705456,28.077726,9.377943e-08
1,2023-11-15,0.5,20.00,87.25,0.307757,-0.239613,-0.096364,0.056723,0.000788,0.651579,0.002323,1.524284,213.016098,8.151818,25.696281,29.530020,28.035089,1.243761e-06
2,2023-11-15,0.5,20.00,87.50,0.361767,-0.274904,-0.155237,0.061546,0.000665,0.060814,0.013914,1.535927,209.882004,8.116321,30.314779,32.050991,28.162968,4.969929e-06
3,2023-11-15,0.5,20.00,87.75,0.269444,-0.050030,0.156805,0.052620,0.000643,0.023415,0.045637,1.560925,207.318329,8.058519,32.787670,36.222424,28.436258,-7.740124e-07
4,2023-11-15,0.5,20.00,88.00,0.260333,-0.017039,0.178411,0.052023,0.000686,0.022156,0.056481,1.668919,206.241653,8.051982,32.575329,36.790691,28.268316,-6.153559e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163675,2025-11-24,0.5,21.75,91.00,6.086784,0.019307,-0.006726,0.354040,0.006740,67.023109,0.007953,51.056564,292.539551,8.476734,4.108316,14.158175,26.029886,-2.036620e-07
163690,2025-11-24,0.5,22.00,90.50,4.213870,0.036492,0.022951,0.264383,0.008627,24.784285,0.006856,13.655972,297.137207,8.379571,7.059252,11.646044,26.047014,5.422925e-07
163691,2025-11-24,0.5,22.00,90.75,5.841084,0.052426,-0.003523,0.332640,0.008557,42.097630,0.007924,25.359039,315.378754,8.417076,6.135298,12.288780,26.050995,-2.651847e-07
163709,2025-11-24,0.5,22.25,91.00,6.891112,0.013623,-0.001812,0.371102,0.009915,80.340973,0.117012,34.292389,394.874207,8.521251,0.835270,14.133733,26.278051,-1.312876e-08


In [13]:
# Check unique depth values
if "depth" in df.columns:
    unique_depths = df["depth"].unique()
    print("Unique depth values:", unique_depths)

    # If all depth values are exactly 0.5 → drop the whole column
    if len(unique_depths) == 1 and unique_depths[0] == 0.5:
        print("All depth values are 0.5 → dropping depth column.")
        df = df.drop(columns=["depth"])
    else:
        print("Depth column has multiple values → keeping it.")

df

Unique depth values: [0.5]
All depth values are 0.5 → dropping depth column.


,time,latitude,longitude,chl,uo,vo,kd,fe,no3,po4,si,o2,ph,so,spco2,thetao,wo
0,2023-11-15,20.00,87.00,0.281021,-0.006259,-0.022032,0.053912,0.001012,1.868117,0.000773,1.615477,213.700699,8.144088,25.942343,29.705456,28.077726,9.377943e-08
1,2023-11-15,20.00,87.25,0.307757,-0.239613,-0.096364,0.056723,0.000788,0.651579,0.002323,1.524284,213.016098,8.151818,25.696281,29.530020,28.035089,1.243761e-06
2,2023-11-15,20.00,87.50,0.361767,-0.274904,-0.155237,0.061546,0.000665,0.060814,0.013914,1.535927,209.882004,8.116321,30.314779,32.050991,28.162968,4.969929e-06
3,2023-11-15,20.00,87.75,0.269444,-0.050030,0.156805,0.052620,0.000643,0.023415,0.045637,1.560925,207.318329,8.058519,32.787670,36.222424,28.436258,-7.740124e-07
4,2023-11-15,20.00,88.00,0.260333,-0.017039,0.178411,0.052023,0.000686,0.022156,0.056481,1.668919,206.241653,8.051982,32.575329,36.790691,28.268316,-6.153559e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163675,2025-11-24,21.75,91.00,6.086784,0.019307,-0.006726,0.354040,0.006740,67.023109,0.007953,51.056564,292.539551,8.476734,4.108316,14.158175,26.029886,-2.036620e-07
163690,2025-11-24,22.00,90.50,4.213870,0.036492,0.022951,0.264383,0.008627,24.784285,0.006856,13.655972,297.137207,8.379571,7.059252,11.646044,26.047014,5.422925e-07
163691,2025-11-24,22.00,90.75,5.841084,0.052426,-0.003523,0.332640,0.008557,42.097630,0.007924,25.359039,315.378754,8.417076,6.135298,12.288780,26.050995,-2.651847e-07
163709,2025-11-24,22.25,91.00,6.891112,0.013623,-0.001812,0.371102,0.009915,80.340973,0.117012,34.292389,394.874207,8.521251,0.835270,14.133733,26.278051,-1.312876e-08


In [14]:
# ------------------------------------------------------------
# CHECK FOR DUPLICATES USING PRIMARY KEY (time, lat, lon)
# ------------------------------------------------------------
pk = ["time", "latitude", "longitude"]

# Detect duplicates
dup_mask = df.duplicated(subset=pk, keep=False)
duplicate_rows = df[dup_mask]

if len(duplicate_rows) > 0:
    print("\nDUPLICATES FOUND!")
    print("Number of duplicate rows:", len(duplicate_rows))
    print("\nSample duplicates:")
    print(duplicate_rows.head())

    # --------------------------------------------------------
    # OPTION A — AVERAGE duplicates (recommended)
    # --------------------------------------------------------
    print("\nAveraging duplicate rows based on primary key...")
    df = df.groupby(pk).mean().reset_index()

else:
    print("\nNo duplicates found. Primary key is unique.")


No duplicates found. Primary key is unique.


In [15]:
df.to_csv("data.csv", index=False)